# Лабораторная работа №9: Fine-tuning VLM с помощью LLaMA-Factory

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.


## Цели обучения

После работы студент должен уметь:

- Преобразовать мультимодальные данные в instruction format.
- Сформировать QLoRA YAML и запустить SFT.
- Сравнить base и adapter, включая OOD slice.


## Теоретическая часть

SFT минимизирует autoregressive loss только на целевом ответе. QLoRA хранит base weights в 4-bit NF4 и обучает low-rank update $\Delta W=BA$, что уменьшает память. LLaMA-Factory фиксирует pipeline в YAML-конфигурации.


### Математическая постановка

$$W'=W_q+\frac{\alpha}{r}BA,\quad B\in\mathbb R^{d\times r},\ A\in\mathbb R^{r\times k}.$$ Обучаются только $A,B$; сравнение проводится с той же base model и evaluation set.


### Материалы

- [LLaMA-Factory](https://github.com/hiyouga/LLaMA-Factory)
- [QLoRA](https://arxiv.org/abs/2305.14314)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практическое задание

1. Преобразовать мультимодальные данные в instruction format.
2. Сформировать QLoRA YAML и запустить SFT.
3. Сравнить base и adapter, включая OOD slice.

**Обязательные артефакты:** код, конфигурация, метрики, минимум одна контролируемая ablation, примеры ошибок и краткие выводы.


### Профиль вычислений

- **Основной режим:** поддерживаемая VLM 4B NF4, LoRA r=8.
- **Ограничение данных:** 100–500 instruction examples, 336–448 px.
- **Fallback:** VLM 2B NF4, тот же dataset/config.
- Все длительные этапы должны сохранять checkpoint и продолжаться после перезапуска. Оценка не зависит от размера модели: важны корректность эксперимента и выводы.


## Реализация


In [ ]:
# Базовое воспроизводимое окружение
from pathlib import Path
import json, os, random, time
import numpy as np

SEED = 42
FAST_DEV_RUN = True  # False — полный эксперимент
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch = None
    DEVICE = "cpu"
# Датасеты курса лежат в общем каталоге <корень курса>/data:
# notebook выполняется из labs/lab_XX_*/, поэтому корень курса — cwd.parent.parent.
# Для локальных запусков вне labs/ также поднимаемся, пока не найдём каталог data с CIFAR.
_CANDIDATES = [Path.cwd().parent.parent / "data", Path.cwd().parent / "data", Path.cwd() / "data"]
DATA_ROOT = Path(os.getenv("CV_COURSE_DATA") or next((c for c in _CANDIDATES if (c / "cifar-10-batches-py").exists()), _CANDIDATES[0]))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print({"device": DEVICE, "fast_dev_run": FAST_DEV_RUN, "data_root": str(DATA_ROOT)})


In [ ]:
def check(condition, message):
    """Мини-проверка в стиле YAAM: молчит, если всё хорошо; печатает, если найдена проблема."""
    if condition:
        print(f"✔ {message}")
    else:
        print(f"✘ ВНИМАНИЕ: {message}")

### Валидация записи обучающего набора VLM

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. Реализуйте необходимые функции: `validate_record(r)`.
2. Выполните требуемые вычисления одноразовым сценарием; все случайные величины фиксируйте через ранее объявленный `SEED`.
3. Выведите итоговые значения в ячейке вывода отдельной инструкцией вывода — эти значения используются в последующих заданиях.
4. Оцените результат на непротиворечивость: значения конечны, непусты и согласуются с постановкой; при отклонениях разберитесь, откуда они взялись, до того как двигаться дальше.

**Результат.** Проверенные значения в ячейке вывода; значения используются в последующих заданиях.

**Критерий принятия.** Задание считается принятым, если ячейка отрабатывает без ошибок, значения конечны и непусты, требуемое сохранено на диск.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: Сохраните сигнатуры и формат артефактов, описанные выше.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Подготовка адаптера VLM

**Постановка.** Реализация, полученная в настоящем задании, образует вычислительное ядро практикума: от её корректности зависит валидность всех последующих измерений.

Выполните следующие действия:

1. Выполните требуемые вычисления одноразовым сценарием; все случайные величины фиксируйте через ранее объявленный `SEED`.
2. Сохраните результат в `artifacts/lab9_adapter` и продублируйте ключевые числа в ячейке вывода отдельной инструкцией вывода.
3. Оцените результат на непротиворечивость: значения конечны, непусты и согласуются с постановкой; при отклонениях разберитесь, откуда они взялись, до того как двигаться дальше.

**Результат.** Проверенные значения в ячейке вывода и `artifacts/lab9_adapter`.

**Критерий принятия.** Задание считается принятым, если ячейка отрабатывает без ошибок, значения конечны и непусты, требуемое сохранено на диск.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/lab9_adapter
# Артефакт: Сохраните сигнатуры и формат артефактов, описанные выше.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Команда запуска LLaMA-Factory и фиксация ресурсов

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. После установки LLaMA-Factory запуск выполняется командой.
2. Не запускайте до замены model/template и проверки поддержки текущей версией.
3. Сохраните stdout, nvidia-smi peak memory и adapter directory.

**Результат.** Результаты задания в ячейке вывода.

**Критерий принятия.** Задание считается принятым, если все пункты плана исполнены в этом ноутбуке: ячейка отрабатывает без ошибок, вывод виден в ячейке вывода, требуемое сохранено.

In [ ]:
# Реализуйте это задание по постановке выше.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Попарное сравнение базовой и адаптированной модели

*Вес: 0.75 балла.*

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. Реализуйте необходимые функции: `paired_report(base_rows, adapted_rows)`.
2. Выполните требуемые вычисления одноразовым сценарием; все случайные величины фиксируйте через ранее объявленный `SEED`.
3. Выведите итоговые значения в ячейке вывода отдельной инструкцией вывода — эти значения используются в последующих заданиях.
4. Оцените результат на непротиворечивость: значения конечны, непусты и согласуются с постановкой; при отклонениях разберитесь, откуда они взялись, до того как двигаться дальше.

**Результат.** Проверенные значения в ячейке вывода; значения используются в последующих заданиях.

**Критерий принятия.** Задание считается принятым, если ячейка отрабатывает без ошибок, значения конечны и непусты, требуемое сохранено на диск.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: Сохраните сигнатуры и формат артефактов, описанные выше.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Свип learning rate в конфигурации обучения

*Вес: 0.75 балла.*

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. Убедитесь, что отличия только в learning_rate.

**Результат.** Для lr [5e-5, 1e-4, 3e-4] сгенерируйте три варианта конфига (только поле learning_rate) и запишите их в artifacts/lr_sweeps.yaml каждый отдельно.

**Критерий принятия.** Задание считается принятым, если все пункты плана выполнены, результат виден в ячейке вывода и сохранён на диск.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: Для lr [5e-5, 1e-4, 3e-4] сгенерируйте три варианта конфига (только поле learning_rate) и запишите их в artifacts/lr_sweeps.yaml каждый отдельно.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Проверка записей train.json

*Вес: 0.75 балла.*

**Постановка.** Разделение подготовки данных от моделирования — базовое требование к воспроизводимым экспериментам: детерминированная загрузка, фиксированные преобразования и контролируемая генерация батчей позволяют атрибутировать изменение метрик изменениям модели, а не случайным различиям пайплайна.

Выполните следующие действия:

1. Прочитайте data/lab9/train.json, для каждой записи выполните validate_record; общее число записей и ошибок — в artifacts/dataset_report.json.

**Критерий принятия.** Задание считается принятым, если прочитайте data/lab9/train.json, для каждой записи выполните validate_record; общее число записей и ошибок — в artifacts/dataset_report.json.
.

In [ ]:
# Реализуйте это задание по постановке выше.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Эквивалентные конфигурации effective batch

*Вес: 0.75 балла.*

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. Для вариантов (bs,accum) [(1,16),(2,8),(4,4)] вычислите effective batch; все должны быть 16.

**Результат.** Сохраните artifacts/effective_batch.json.

**Критерий принятия.** Задание считается принятым, если все пункты плана выполнены, результат виден в ячейке вывода и сохранён на диск.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: Сохраните artifacts/effective_batch.json.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Оценка числа шагов обучения по конфигу

*Вес: 0.75 балла.*

**Постановка.** Реализация, полученная в настоящем задании, образует вычислительное ядро практикума: от её корректности зависит валидность всех последующих измерений.

Выполните следующие действия:

1. Для конфига оцените примерное число шагов = ceil(max_samples / (bs*accum)) * epochs; убедитесь, что при FAST_DEV_RUN шагов <= 8.

**Результат.** Сохраните artifacts/step_estimate.json.

**Критерий принятия.** Задание считается принятым, если все пункты плана выполнены, результат виден в ячейке вывода и сохранён на диск.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: Сохраните artifacts/step_estimate.json.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Аудит обучаемых параметров

*Вес: 0.5 балла.*

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. Выполните требуемые вычисления одноразовым сценарием; все случайные величины фиксируйте через ранее объявленный `SEED`.
2. Выведите итоговые значения в ячейке вывода отдельной инструкцией вывода — эти значения используются в последующих заданиях.
3. Оцените результат на непротиворечивость: значения конечны, непусты и согласуются с постановкой; при отклонениях разберитесь, откуда они взялись, до того как двигаться дальше.

**Результат.** Проверенные значения в ячейке вывода; значения используются в последующих заданиях.

**Критерий принятия.** Задание считается принятым, если ячейка отрабатывает без ошибок, значения конечны и непусты, требуемое сохранено на диск.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: Соберите: число записей, число шагов, effective batch, пути артефактов — в artifacts/sft_summary.json (assert на ключи).

raise NotImplementedError("Реализуйте это задание в этой ячейке")

In [ ]:
# Самопроверка: все обязательные ячейки выше должны быть исполнены (кроме бонусной), в outputs не должно быть ошибок.
# Если какая-то ячейка не исполнена или упала - вернитесь и доведите её до конца.
check(True, "самопроверка: обязательные ячейки исполнены, ошибок нет")

### Инженерия результатов

Ниже — усложнённые задания обязательной части. Они не повторяют базовые, а достраивают вокруг результата лабы инженерный контур: стресс-тест, статистическая значимость, воспроизводимость, стоимость и сводный дашборд. Выполняются по порядку: последнее собирает итог из предыдущих. Вес каждого — 1 балл.

### Стресс-тест: качество при возмущении входа

**Вес: 1 балл, обязательное.** Возьмите основной результат лабы как «счётчик качества» и постройте кривую его деградации при нарастающих возмущениях (несколько уровней, повторы на каждый). Ключевой вопрос — порог: при какой силе возмущения результат перестаёт быть пригодным. Артефакт: `artifacts/stress_robustness.json`.

In [ ]:
# Контекст лабы: lab 09 llamafactory vlm sft. Счётчик качества — число из результатов этой лабы (метрика, IoU, accuracy, качественная оценка).
# Что нужно сделать:
# 1) определить количественный счётчик качества по результатам лабы;
# 2) построить его зависимость от силы возмущения входа (3-5 уровней, по 3 повтора на уровень);
# 3) записать кривую (сила, среднее, std) и относительную деградацию;
# 4) вывод: при какой силе возмущения результат ещё приемлем.
# Артефакт: artifacts/stress_robustness.json
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Доверительные интервалы: бутстреп и значимость разницы

**Вес: 1 балл, обязательное.** Одного среднего недостаточно: превратите результат в выборку испытаний, постройте 95% бутстреп-интервал для основной конфигурации и для ослабленной, проверьте, пересекаются ли интервалы. Отдельно посмотрите, как ширина интервала зависит от числа ресэплов. Артефакт: `artifacts/bootstrap_ci.json`.

In [ ]:
# Контекст лабы: lab 09 llamafactory vlm sft.
# Что нужно сделать:
# 1) превратить основной результат в выборку испытаний (успех/неудача) или числовых значений;
# 2) бутстреп (>=400 ресэплов) 95% CI для основной и для ослабленной конфигурации (например, x0.9);
# 3) пересекаются ли интервалы - вывод о значимости разницы;
# 4) бонус-наблюдение: как ширина CI зависит от числа ресэплов (100/400/1600).
# Артефакт: artifacts/bootstrap_ci.json
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Воспроизводимость: разброс по случайным зернам (seed)

**Вес: 1 балл, обязательное.** Повторите ключевой эксперимент лабы при нескольких состояниях генератора случайных чисел (если полный запуск дорогой — на подвыборке), оцените стандартное отклонение и ответьте: можно ли доверять одиночному запуску. Артефакт: `artifacts/seed_variance.json`.

In [ ]:
# Контекст лабы: lab 09 llamafactory vlm sft.
# Что нужно сделать:
# 1) повторить ключевой эксперимент лабы при 4 разных seed (если дорого - на подвыборке);
# 2) записать пары (seed, score) и посчитать стандартное отклонение;
# 3) вывод: можно ли доверять одиночному запуску; что конкретно улучшит воспроизводимость (фиксация генераторов, детерминизм,фиксация генераторов...);
# 4) отдельно: какие операции в лабе недетерминированы.
# Артефакт: artifacts/seed_variance.json
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Стоимость: время, память и вес артефактов

**Вес: 1 балл, обязательное.** Замерьте, во что обходится результат: время ключевого прогона, пик памяти (tracemalloc), суммарный вес артефактов. Найдите главный вкладчик и постройте оценку, как стоимость растёт при десятикратном увеличении данных. Артефакт: `artifacts/cost_profile.json`.

In [ ]:
# Контекст лабы: lab 09 llamafactory vlm sft.
# Что нужно сделать:
# 1) замерить время ключевого прогона, пик памяти (tracemalloc) и суммарный вес артефактов;
# 2) найти самого тяжёлого вкладчика (по времени/памяти/размеру);
# 3) прогноз: как стоимость растёт при увеличении данных в 10 раз (по какому компоненту - линейно, квадратично);
# 4) вывод: главное вычислительное узкое место лабы.
# Артефакт: artifacts/cost_profile.json
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Сводный дашборд: все проверки лабы в одном отчёте

**Вес: 1 балл, обязательное.** Соберите результаты всех предыдущих проверок и артефакты лабы в один отчёт: словарь флагов PASS/WARN/MISSING по каждому направлению, машиночитаемый JSON и человекочитаемый markdown. Это итоговая самодиагностика лабораторной. Артефакты: `artifacts/final_dashboard.json`, `artifacts/final_dashboard.md`.

In [ ]:
# Контекст лабы: lab 09 llamafactory vlm sft. Задание собирает итог из предыдущих - выполняется последним.
# Что нужно сделать:
# 1) прочитать json-результаты всех проверок выше (и остальные артефакты лабы);
# 2) выставить по каждому направлению флаг PASS/WARN/MISSING с порогами, которые вы объявите явно;
# 3) сохранить artifacts/final_dashboard.json (машиночитаемый) и artifacts/final_dashboard.md (человекочитаемый);
# 4) резюме лабы в трёх строках: что работает, что осторожно, что не проверено.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Аудит файлов артефактов и их размеров

*Вес: 0.5 балла.*

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. Соберите список всех файлов в каталоге artifacts/, для каждого — размер в байтах; убедитесь, что ни один не пуст.

**Результат.** Сохраните artifacts/files_audit.json.

**Критерий принятия.** Задание считается принятым, если все пункты плана выполнены, результат виден в ячейке вывода и сохранён на диск.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: Сохраните artifacts/files_audit.json.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Мини-исследование: гипотеза, эксперимент и вывод

*Вес: 0.5 балла.*

**Постановка.** Сравнение конфигураций корректно только при прочих равных: идентичном числе шагов оптимизации, наборе данных и протоколе оценки. При нарушении этих условий различия метрик объясняются параметрами эксперимента, а не сравниваемыми методами.

Выполните следующие действия:

1. Гипотеза.
2. Эксперимент с фиксированным seed.
3. Метрика и сравнение.
4. Вывод: подтвердилась/нет, почему.

**Критерий принятия.** Задание считается принятым, если вывод: подтвердилась/нет, почему, результат виден в ячейке вывода в этом ноутбуке и сохранён на диск.

In [ ]:
# Реализуйте это задание по постановке выше.

raise NotImplementedError("Реализуйте это задание в этой ячейке")

## Анализ результатов

Заполните после запуска:

1. Таблица сравниваемых конфигураций и вычислительный бюджет.
2. Основные метрики с доверительными интервалами или вариативностью по seed, где это возможно.
3. Не менее пяти характерных ошибок/неудачных примеров.
4. Ablation: какой компонент действительно дал эффект?
5. Ограничения эксперимента и следующий проверяемый шаг.


In [ ]:
# Автоматическая проверка наличия каталога артефактов
assert ARTIFACTS.exists()
print("Artifacts:", [p.name for p in ARTIFACTS.iterdir()])


## Выводы

Сформулируйте 3–5 выводов, каждый из которых опирается на измерение или наблюдаемый пример. Не подменяйте вывод перечислением выполненных действий.
